In [1]:
import os
import torch
from torch_geometric.data import Data
import pickle

In [2]:
from torch_geometric.data import Dataset

class GraphPartDataset(Dataset):
    def __init__(self, root_dir):
        super().__init__()
        self.root_dir = root_dir
        self.file_list = sorted([f for f in os.listdir(root_dir) if f.endswith('.pt')])

    def len(self):
        return len(self.file_list)

    def get(self, idx):
        return torch.load(os.path.join(self.root_dir, self.file_list[idx]))


In [3]:
from torch_geometric.loader import DataLoader

graph_dataset = GraphPartDataset(root_dir="/home/iiitb/Desktop/anant/GridRaster/processed_data")
graph_loader = DataLoader(graph_dataset, batch_size=8, shuffle=True)


In [4]:
graph_loader.dataset.__len__()

4319

In [5]:
from torch_geometric.data import Batch

for batch_data in graph_loader:
    print(batch_data)
    break


# Split batched data into list of individual Data objects
individual_graphs = batch_data.to_data_list()

for i, data in enumerate(individual_graphs):
    print(f"Graph {i} has {data.num_nodes} nodes, label shape: {data.y.shape}")


DataBatch(x=[485, 1024], edge_index=[2, 50529], y=[485], edge_weight=[50529], batch=[485], ptr=[9])
Graph 0 has 42 nodes, label shape: torch.Size([42])
Graph 1 has 23 nodes, label shape: torch.Size([23])
Graph 2 has 124 nodes, label shape: torch.Size([124])
Graph 3 has 153 nodes, label shape: torch.Size([153])
Graph 4 has 36 nodes, label shape: torch.Size([36])
Graph 5 has 3 nodes, label shape: torch.Size([3])
Graph 6 has 89 nodes, label shape: torch.Size([89])
Graph 7 has 15 nodes, label shape: torch.Size([15])


In [6]:
from model import Net

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [ ]:
model = Net(mp_units=[64], mp_act="ELU", in_channels=1024, n_clusters=2).to(device)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = torch.nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss = 0

    for batch in graph_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        out, mc_loss, o_loss = model(batch.x, batch.edge_index, batch.edge_weight, batch.batch)
        loss = criterion(out, batch.y) + mc_loss + o_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss / len(graph_loader):.4f}")


Epoch 1 | Loss: 0.3367
Epoch 2 | Loss: 0.3328
Epoch 3 | Loss: 0.3350
Epoch 4 | Loss: 0.3357
Epoch 5 | Loss: 0.3351
Epoch 6 | Loss: 0.3345
Epoch 7 | Loss: 0.3343
Epoch 8 | Loss: 0.3343
Epoch 9 | Loss: 0.3334
Epoch 10 | Loss: 0.3344


: 